In [ ]:
# UAS DL TASK 1 (MNLI)

## SETUP

In [1]:
!pip install transformers datasets evaluate accelerate scikit-learn -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 140.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 51.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


## Import

In [2]:
import os
os.environ["WANDB_DISABLED"] = "true"

import torch
import numpy as np
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset

MODEL_CKPT = "distilbert-base-uncased"
BATCH_SIZE = 64
EPOCHS = 3

print(f"GPU Available: {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name(0)}")

GPU Available: True
Device Name: NVIDIA L4


## Load Data & Cek Output

In [3]:
# Load GLUE MNLI
dataset = load_dataset("glue", "mnli")

print("Label List:", dataset['train'].features['label'].names)
print("Contoh Data:", dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

Label List: ['entailment', 'neutral', 'contradiction']
Contoh Data: {'premise': 'Conceptually cream skimming has two basic dimensions - product and geography.', 'hypothesis': 'Product and geography are what make cream skimming work. ', 'label': 1, 'idx': 0}


## Tokenizer

In [4]:
MODEL_CKPT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def preprocess(examples):
    # Gabungkan Premise + Hypothesis
    return tokenizer(examples["premise"], examples["hypothesis"], truncation=True, padding=True)

tokenized_ds = dataset.map(preprocess, batched=True)

print("Keys available:", tokenized_ds['train'][0].keys())

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/392702 [00:00<?, ? examples/s]

Map:   0%|          | 0/9815 [00:00<?, ? examples/s]

Map:   0%|          | 0/9832 [00:00<?, ? examples/s]

Map:   0%|          | 0/9796 [00:00<?, ? examples/s]

Map:   0%|          | 0/9847 [00:00<?, ? examples/s]

Keys available: dict_keys(['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'attention_mask'])


## Model & Training

In [6]:
id2label = {0: "ENTAILMENT", 1: "NEUTRAL", 2: "CONTRADICTION"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT, num_labels=3, id2label=id2label, label2id=label2id
)

# Metric Accuracy
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

args = TrainingArguments(
    output_dir="./mnli_model",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,


    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    save_total_limit=2,


    load_best_model_at_end=True,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation_matched"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting Training MNLI...")
trainer.train()
trainer.save_model("./final_mnli")



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2113603554.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting Training MNLI...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.526300,0.514228,0.795008
2,0.443300,0.486111,0.811717
3,0.364500,0.495962,0.817626


## Testing

In [8]:
model.eval()

# Pasangan kalimat (Premise, Hypothesis)
mnli_samples = [
    # CONTOH 1: ENTAILMENT (Nyambung/Benar)
    ("A soccer player is running across the field.", "A person is playing sports."),

    # CONTOH 2: CONTRADICTION (Bertolak Belakang)
    ("A man is inspecting the uniform of a figure in some East Asian country.", "The man is sleeping on the couch."),

    # CONTOH 3: NEUTRAL (Tidak Berhubungan/Tidak Tahu)
    ("The wedding party took pictures inside the building.", "The photos were the best ever taken."),

    # CONTOH 4: Test Sendiri
    ("Messi is the GOAT.", "Ronaldo is the GOAT.")
]

print("=== HASIL PREDIKSI MNLI ===")
print("Labels: 0=Entailment, 1=Neutral, 2=Contradiction\n")

for premise, hypothesis in mnli_samples:
    # 1. Tokenisasi DUA KALIMAT (PENTING!)
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to("cuda")

    # 2. Prediksi
    with torch.no_grad():
        logits = model(**inputs).logits

    # 3. Ambil nilai terbesar (Argmax) karena ini Single Label
    pred_id = torch.argmax(logits, dim=1).item()

    # 4. Translate ke Label
    # Menggunakan mapping id2label dari config model
    pred_label = model.config.id2label[pred_id]

    print(f"Premise   : {premise}")
    print(f"Hypothesis: {hypothesis}")
    print(f"Prediksi  : {pred_label} (ID: {pred_id})\n")

=== HASIL PREDIKSI MNLI ===
Labels: 0=Entailment, 1=Neutral, 2=Contradiction

Premise   : A soccer player is running across the field.
Hypothesis: A person is playing sports.
Prediksi  : ENTAILMENT (ID: 0)

Premise   : A man is inspecting the uniform of a figure in some East Asian country.
Hypothesis: The man is sleeping on the couch.
Prediksi  : CONTRADICTION (ID: 2)

Premise   : The wedding party took pictures inside the building.
Hypothesis: The photos were the best ever taken.
Prediksi  : NEUTRAL (ID: 1)

Premise   : Messi is the GOAT.
Hypothesis: Ronaldo is the GOAT.
Prediksi  : CONTRADICTION (ID: 2)

